# Lung preprocessing QC

This notebook is **read-only QC** for `data/lung/lung_benchmark_ready.h5ad`. It does not perform preprocessing or train representations. The canonical preprocessing run is `python scripts/preprocess.py --config configs/preprocessing/lung.yaml`.

In [1]:
from pathlib import Path
import json

import anndata as ad
import pandas as pd

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "src" / "scrna_benchmark").is_dir():
            return p
    raise RuntimeError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
ADATA_PATH = REPO_ROOT / "data/lung/lung_benchmark_ready.h5ad"
QC_DIR = REPO_ROOT / "results/preprocessing/lung"
SUMMARY_PATH = QC_DIR / "preprocessing_summary.json"
print(REPO_ROOT)

/users/xchen5/scRNA-cross-donor-generalization


## 1. Frozen-object invariants

In [2]:
if not ADATA_PATH.exists():
    raise FileNotFoundError(ADATA_PATH)
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(SUMMARY_PATH)

adata = ad.read_h5ad(ADATA_PATH, backed="r")
with SUMMARY_PATH.open() as f:
    summary = json.load(f)

observed = {
    "cells": adata.n_obs,
    "HVGs": adata.n_vars,
    "donors": adata.obs["donor_id"].astype(str).nunique(),
    "cell_types": adata.obs["cell_type"].astype(str).nunique(),
    "batches": adata.obs["batch"].astype(str).nunique(),
    "PCA_dims": adata.obsm["X_pca"].shape[1],
    "Harmony_dims": adata.obsm["X_harmony"].shape[1],
    "scVI_dims": adata.obsm["X_scVI"].shape[1],
    "has_counts_layer": "counts" in adata.layers,
}
display(pd.DataFrame([observed]))

assert observed["cells"] == 83966
assert observed["HVGs"] == 1000
assert observed["donors"] == 104
assert observed["cell_types"] == 31
assert observed["batches"] == 13
assert observed["PCA_dims"] == 15
assert observed["Harmony_dims"] == 15
assert observed["scVI_dims"] == 15
assert observed["has_counts_layer"]

# Legacy lung preparation retained 22,921 genes after gene filtering, before HVG restriction.
assert summary["stages"]["after_gene_filtering"]["n_genes"] == 22921
print("All frozen-object checks passed.")

,cells,HVGs,donors,cell_types,batches,PCA_dims,Harmony_dims,scVI_dims,has_counts_layer
0,83966,1000,104,31,13,15,15,15,True


All frozen-object checks passed.


## 2. Preprocessing stage summary

In [3]:
stage_df = pd.DataFrame(summary["stages"]).T
stage_df.index.name = "stage"
display(stage_df)
display(pd.DataFrame(summary["expected_checks"]).T)

,n_cells,n_genes,n_donors,n_celltypes
stage,,,,
loaded,584944,27402,107,50
after_dataset_filters,441942,27402,104,38
after_celltype_support_filter,439931,27402,104,31
after_downsampling,83966,27402,104,31
after_gene_filtering,83966,22921,104,31
benchmark_ready,83966,1000,104,31


,expected,observed,ok
n_cells,83966,83966,True
n_donors,104,104,True
n_celltypes,31,31,True
n_hvg,1000,1000,True


## 3. Cell-type eligibility and final composition

In [4]:
support_before = pd.read_csv(QC_DIR / "celltype_support_before_filter.csv")
support_final = pd.read_csv(QC_DIR / "celltype_support_final.csv")
celltype_counts = pd.read_csv(QC_DIR / "celltype_counts_final.csv")

display(support_before)
display(celltype_counts)
print("Final support table (informational after downsampling):")
display(support_final)

,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,Alveolar macrophages,64229,81,True,True,True
1,Suprabasal,40448,74,True,True,True
2,Basal resting,38717,74,True,True,True
3,Multiciliated,38675,99,True,True,True
4,Goblet,38449,82,True,True,True
5,Club,35972,79,True,True,True
6,Interstitial macrophages,32066,88,True,True,True
7,CD8 T cells,26699,83,True,True,True
8,CD4 T cells,18433,85,True,True,True
9,Classical monocytes,16537,84,True,True,True


,cell_type,n_cells
0,Multiciliated,6864
1,Interstitial macrophages,5470
2,Classical monocytes,5291
3,Alveolar macrophages,5074
4,DC2,4829
5,CD8 T cells,4761
6,CD4 T cells,4175
7,Suprabasal,3915
8,Club,3724
9,Goblet,3614


Final support table (informational after downsampling):


,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,Multiciliated,6864,99,True,True,True
1,Interstitial macrophages,5470,88,True,True,True
2,Classical monocytes,5291,84,True,True,True
3,Alveolar macrophages,5074,81,True,True,True
4,DC2,4829,88,True,True,True
5,CD8 T cells,4761,83,True,True,True
6,CD4 T cells,4175,85,True,True,True
7,Suprabasal,3915,74,True,True,True
8,Club,3724,79,True,True,True
9,Goblet,3614,82,True,True,True


## 4. Donor and source-dataset composition

In [5]:
donor_counts = pd.read_csv(QC_DIR / "donor_counts_final.csv")
display(donor_counts.describe())

batch_counts = (
    adata.obs["batch"].astype(str).value_counts()
    .rename_axis("batch").rename("n_cells").reset_index()
)
display(batch_counts)

donor_by_batch = pd.crosstab(
    adata.obs["donor_id"].astype(str),
    adata.obs["batch"].astype(str),
)
display(donor_by_batch.head())

,n_cells
count,104.000000
mean,807.365385
std,432.906390
min,7.000000
25%,495.000000
50%,674.500000
75%,1084.000000
max,1937.000000


,batch,n_cells
0,Banovich_Kropski_2020,29097
1,Nawijn_2021,11288
2,Barbry_Leroy_2020,9865
3,Lafyatis_Rojas_2019_10Xv2,4785
4,Misharin_Budinger_2018,4748
5,Meyer_2019,4059
6,Seibold_2020_10Xv3,3996
7,Jain_Misharin_2021_10Xv2,3582
8,Misharin_2021,3550
9,Teichmann_Meyer_2019,3295


batch,Banovich_Kropski_2020,Barbry_Leroy_2020,Jain_Misharin_2021_10Xv1,Jain_Misharin_2021_10Xv2,Lafyatis_Rojas_2019_10Xv1,Lafyatis_Rojas_2019_10Xv2,Meyer_2019,Misharin_2021,Misharin_Budinger_2018,Nawijn_2021,Seibold_2020_10Xv2,Seibold_2020_10Xv3,Teichmann_Meyer_2019
donor_id,,,,,,,,,,,,,
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_4837472020-3173-NC002,0,0,669,0,0,0,0,0,0,0,0,0,0
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_4837472020-3173-NC003,0,0,0,954,0,0,0,0,0,0,0,0,0
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_4837472020-3173-NC004,0,0,0,779,0,0,0,0,0,0,0,0,0
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_4837472020-3173-NC005,0,0,0,824,0,0,0,0,0,0,0,0,0
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_4837472020-3173-NC006,0,0,0,522,0,0,0,0,0,0,0,0,0


## 5. Provenance artifacts

In [6]:
for name in [
    "selected_cells.csv",
    "hvg_genes.csv",
    "resolved_config.yaml",
    "software_versions.json",
]:
    path = QC_DIR / name
    print(name, "OK" if path.exists() else "MISSING")

print("Selected cells:", len(pd.read_csv(QC_DIR / "selected_cells.csv")))
print("HVGs:", len(pd.read_csv(QC_DIR / "hvg_genes.csv")))

if hasattr(adata, "file") and adata.file is not None:
    adata.file.close()

selected_cells.csv OK
hvg_genes.csv OK
resolved_config.yaml OK
software_versions.json OK
Selected cells: 83966
HVGs: 1000
